# World Bank data download code

In [ ]:
# (1) Paths, country-year table, and variables
import os
import time
from typing import Dict

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PROJECT_ROOT = os.path.dirname(os.getcwd())
ISO_PATH = os.path.join(PROJECT_ROOT, "data", "iso36.csv")
LAST_SURVEY_YEAR_PATH = os.path.join(PROJECT_ROOT, "results", "model_prep", "last_survey_year_by_country.csv")
OUT_DIR = os.path.join(PROJECT_ROOT, "data", "wb")
OUT_PATH = os.path.join(OUT_DIR, "wb_last_survey_year.csv")
LE_OUT_PATH = os.path.join(PROJECT_ROOT, "data", "wb", "wb_life_expectancy_last_survey_year.csv")

INDICATORS: Dict[str, str] = {
    "SP.POP.TOTL": "total_population",
    "SP.POP.65UP.TO.ZS": "age65_share",
}

LE_INDICATORS: Dict[str, str] = {
    "SP.DYN.LE00.IN": "le",
    "SP.DYN.LE00.FE.IN": "female",
    "SP.DYN.LE00.MA.IN": "male",
}

os.makedirs(OUT_DIR, exist_ok=True)

iso = pd.read_csv(ISO_PATH).rename(columns={"isocountry_c": "country"})
last_year = pd.read_csv(LAST_SURVEY_YEAR_PATH)
iso["country"] = iso["country"].astype(str).str.strip()
iso["iso3c"] = iso["iso3c"].astype(str).str.strip().str.upper()
last_year["country"] = last_year["country"].astype(str).str.strip()
last_year["last_survey_year"] = pd.to_numeric(last_year["last_survey_year"], errors="coerce")

country_year = iso[["country", "iso3c", "dataset"]].merge(last_year, on="country", how="left")
missing_year = country_year.loc[country_year["last_survey_year"].isna(), "country"].tolist()
if missing_year:
    raise ValueError(f"Countries in iso36 without last_survey_year: {missing_year}")
country_year["year"] = country_year["last_survey_year"].round().astype(int)

COUNTRY_CODES = sorted(country_year["iso3c"].dropna().unique())
START_YEAR = int(country_year["year"].min())
END_YEAR = int(country_year["year"].max())

print("Countries:", len(COUNTRY_CODES))
print("Years needed:", f"{START_YEAR}-{END_YEAR}")
print("OUT_PATH:", OUT_PATH)
print("LE_OUT_PATH:", LE_OUT_PATH)
country_year.head()

In [12]:
# (2) World Bank API helpers
WB_BASE = "https://api.worldbank.org/v2"

session = requests.Session()
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))


def fetch_indicator_for_countries(code: str, countries, start_year: int, end_year: int, sleep: float = 0.2) -> pd.DataFrame:
    country_expr = ";".join(countries)
    all_rows = []
    page = 1

    while True:
        url = f"{WB_BASE}/country/{country_expr}/indicator/{code}"
        params = {
            "format": "json",
            "per_page": 20000,
            "date": f"{start_year}:{end_year}",
            "page": page,
        }
        resp = session.get(url, params=params, timeout=(10, 180))
        resp.raise_for_status()
        data = resp.json()
        if not isinstance(data, list) or len(data) < 2:
            break

        meta, rows = data[0], data[1]
        if not rows:
            break

        all_rows.extend(rows)
        if page >= int(meta.get("pages", 1)):
            break
        page += 1
        time.sleep(sleep)

    df = pd.DataFrame(all_rows)
    if df.empty:
        return df

    out = df[["countryiso3code", "country", "date", "value"]].copy()
    out["country"] = out["country"].apply(lambda x: x.get("value") if isinstance(x, dict) else x)
    out["date"] = pd.to_numeric(out["date"], errors="coerce").astype("Int64")
    out = out.rename(columns={"countryiso3code": "iso3c", "value": code})
    out["iso3c"] = out["iso3c"].astype(str).str.upper()
    return out


def download_wb_wide(indicators: Dict[str, str], out_path: str) -> pd.DataFrame:
    merged = None
    for code in indicators.keys():
        print("Fetching:", code)
        df = fetch_indicator_for_countries(code, COUNTRY_CODES, START_YEAR, END_YEAR)
        if df.empty:
            print("No data for", code)
            continue
        if merged is None:
            merged = df
        else:
            merged = pd.merge(merged, df, on=["iso3c", "country", "date"], how="outer")

    if merged is None:
        raise RuntimeError("No World Bank data downloaded")

    for code, new_name in indicators.items():
        if code in merged.columns:
            merged = merged.rename(columns={code: new_name})

    exact = country_year.merge(merged, left_on=["iso3c", "year"], right_on=["iso3c", "date"], how="left")
    exact = exact.drop(columns=["date"])
    if "country_y" in exact.columns:
        exact = exact.rename(columns={"country_x": "country", "country_y": "wb_country"})

    value_cols = list(indicators.values())
    exact = exact[["country", "iso3c", "dataset", "last_survey_year", "year", "wb_country"] + value_cols]
    exact = exact.sort_values("country").reset_index(drop=True)
    exact.to_csv(out_path, index=False)
    print("Saved:", out_path)
    print("Shape:", exact.shape)
    return exact

In [13]:
# (3) Download required variables at each country's last survey year
wb_year = download_wb_wide(INDICATORS, OUT_PATH)
le_year = download_wb_wide(LE_INDICATORS, LE_OUT_PATH)

print("WB missing counts:")
print(wb_year[list(INDICATORS.values())].isna().sum())
print("Life expectancy missing counts:")
print(le_year[list(LE_INDICATORS.values())].isna().sum())

wb_year.head()

Fetching: SP.POP.TOTL
Fetching: SP.POP.65UP.TO.ZS
Saved: /Users/karwailin/Desktop/OAwalk/data/wb/wb_last_survey_year.csv
Shape: (36, 8)
Fetching: SP.DYN.LE00.IN
Fetching: SP.DYN.LE00.FE.IN
Fetching: SP.DYN.LE00.MA.IN
Saved: /Users/karwailin/Desktop/OAwalk/data/wb/wb_life_expectancy_last_survey_year.csv
Shape: (36, 9)
WB missing counts:
total_population    0
age65_share         0
dtype: int64
Life expectancy missing counts:
le        0
female    0
male      0
dtype: int64


,country,iso3c,dataset,last_survey_year,year,wb_country,total_population,age65_share
0,Australia,AUS,ALSA,2014.0,2014,Australia,23475686,14.634393
1,Austria,AUT,SHARE,2022.0,2022,Austria,9041851,19.789751
2,Belgium,BEL,SHARE,2022.0,2022,Belgium,11680210,19.767193
3,Bulgaria,BGR,SHARE,2022.0,2022,Bulgaria,6465097,21.697297
4,China,CHN,CHARLS,2018.0,2018,China,1402760000,11.516209


In [14]:
# (4) Fetch World 2024 data for global metrics
print("\n=== Fetching World 2024 data ===")
world_row = {
    "country": "World",
    "iso3c": "WLD",
    "dataset": "WB_WORLD",
    "last_survey_year": 2024.0,
    "year": 2024,
    "wb_country": "World",
}

for code, col_name in LE_INDICATORS.items():
    url = f"{WB_BASE}/country/WLD/indicator/{code}"
    params = {
        "format": "json",
        "per_page": 1000,
        "date": "2024",
    }
    try:
        resp = session.get(url, params=params, timeout=(10, 180))
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list) and len(data) >= 2 and data[1]:
            value = pd.to_numeric(data[1][0].get("value"), errors="coerce")
            world_row[col_name] = value
            print(f"  {col_name}: {value}")
        else:
            world_row[col_name] = np.nan
            print(f"  {col_name}: No data")
    except Exception as e:
        print(f"  Warning: Could not fetch {code} for World: {e}")
        world_row[col_name] = np.nan
    time.sleep(0.2)

world_df = pd.DataFrame([world_row])
le_year = pd.concat([world_df, le_year], ignore_index=True)

print("\nWorld 2024 data added to life expectancy file:")
print(world_df)
print(f"\nTotal rows in le_year: {len(le_year)}")
print("First few rows of le_year:")
print(le_year.head())

le_year.to_csv(LE_OUT_PATH, index=False)
print(f"\nSaved updated {LE_OUT_PATH}")


=== Fetching World 2024 data ===
  le: 73.480380292779
  female: 75.9711455767819
  male: 71.1141721801643

World 2024 data added to life expectancy file:
  country iso3c   dataset  last_survey_year  year wb_country        le  \
0   World   WLD  WB_WORLD            2024.0  2024      World  73.48038   

      female       male  
0  75.971146  71.114172  

Total rows in le_year: 37
First few rows of le_year:
     country iso3c   dataset  last_survey_year  year wb_country         le  \
0      World   WLD  WB_WORLD            2024.0  2024      World  73.480380   
1  Australia   AUS      ALSA            2014.0  2014  Australia  82.300000   
2    Austria   AUT     SHARE            2022.0  2022    Austria  81.295122   
3    Belgium   BEL     SHARE            2022.0  2022    Belgium  81.748780   
4   Bulgaria   BGR     SHARE            2022.0  2022   Bulgaria  74.160976   

      female       male  
0  75.971146  71.114172  
1  84.400000  80.300000  
2  83.600000  79.100000  
3  83.900000  79